In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import datetime as dt

from src.data_preparation.input_preparation import prepare_input

In [3]:
df = pd.read_csv("ETT-small/ETTh1.csv")
df_prepped = prepare_input(df)

In [5]:
test_period_cutoff = dt.datetime(2017, 9, 1, 0, 0, 0)

df_train = df[df['date'] < test_period_cutoff]
df_test = df[df['date'] >= test_period_cutoff]

X_train = df_train.drop(columns=['date', 'OT'])
y_train = df_train['OT']

X_test = df_test.drop(columns=['date', 'OT'])
y_test = df_test['OT']

In [18]:
import pmdarima as pm

candidates = [
    ((1,0,1), (1,1,0,24)),
    ((2,0,0), (1,1,0,24)),
    ((2,0,1), (1,1,0,24)),
    ((1,0,0), (1,1,0,24)),
]

results = {}
for order, seasonal_order in candidates:
    m = pm.ARIMA(order=order, seasonal_order=seasonal_order, suppress_warnings=True)
    m.fit(y_train, X_train)
    results[(order, seasonal_order)] = m.aic()
    print(order, seasonal_order, m.aic(), m.bic())
    del m  # free memory before next fit

(1, 0, 1) (1, 1, 0, 24) 32371.398660923012 32450.956085865804
(2, 0, 0) (1, 1, 0, 24) 32373.169706973084 32452.727131915875
(2, 0, 1) (1, 1, 0, 24) 32371.722561989893 32458.5124801093
(1, 0, 0) (1, 1, 0, 24) 32420.97943668009 32493.30436844626


In [28]:
X_train_improved_columns = ['MUFL', 'MULL', 'LUFL']
X_train_improved = X_train[X_train_improved_columns]

In [29]:
from src.model.arima import fit_arima
order = (1, 0, 1)
seasonal_order = (1, 1, 0, 24)
arima_model = fit_arima(y_train, X_train_improved, order, seasonal_order)

In [30]:
arima_model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                                     SARIMAX Results                                     
=========================================================================================
Dep. Variable:                                OT   No. Observations:                10248
Model:             ARIMA(1, 0, 1)x(1, 1, [], 24)   Log Likelihood              -16175.145
Date:                           Sun, 19 Jul 2026   AIC                          32364.289
Time:                                   21:13:36   BIC                          32414.916
Sample:                                        0   HQIC                         32381.407
                                         - 10248                                         
Covariance Type:                             opg                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
MUFL          -0.0167      0.005     -3.196      0.001      -0.027      -0.006
MULL           0.0569      0.016      3.520      0.000       0.025       0.089
LUFL           0.0879      0.020      4.471      0.000       0.049       0.126
ar.L1          0.9478      0.003    371.102      0.000       0.943       0.953
ma.L1         -0.0771      0.007    -11.428      0.000      -0.090      -0.064
ar.S.L24      -0.4708      0.005    -86.788      0.000      -0.481      -0.460
sigma2         1.3847      0.011    126.616      0.000       1.363       1.406
===================================================================================
Ljung-Box (L1) (Q):                   0.00   Jarque-Bera (JB):              9537.07
Prob(Q):                              0.98   Prob(JB):                         0.00
Heteroskedasticity (H):               0.84   Skew:                            -0.27
Prob(H) (two-sided):                  0.00   Kurtosis:                         7.70
===================================================================================

Warnings:
[1] Covariance matrix calculated using the outer product of gradients (complex-step).
"""